<a href="https://colab.research.google.com/github/peremartra/CH13/blob/main/CH13_family_model_S_v1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CH13 — Building a Model Family: Model S

Second step of the cascade. **S** is structurally pruned from **M**, while the
current configuration distills it with the original specialist **T** as teacher.
This is a deliberate distinction in this notebook: M supplies the topology and
pruning source; T supplies the KD targets.

| | T | M | S (this notebook) |
|---|---|---|---|
| Layers | 28 | 25 | 22 |
| MLP intermediate | 3072 | 2304 | resolved below |
| Non-embedding params | measured in T notebook | measured here | measured below |
| Pruning source | — | T | **M** |
| KD teacher | — | T | **T** |

The procedure is: reuse the split, calibrate on the current pruning source,
rank layers, prune depth, prune width, distill, and evaluate across seeds. The
S notebook recomputes layer importance on M rather than reusing T's ranking.

The train/test split is inherited from M's handoff file so the results remain
comparable across the two stages.

## 1. Setup

In [ ]:
!pip install -q \
    transformers==5.0.0 \
    datasets==4.0.0 \
    accelerate==1.12.0 \
    scikit-learn \
    pandas \
    optipfair

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 64.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 494.8/494.8 kB 44.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 380.9/380.9 kB 38.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.7/65.7 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 12.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.6/193.6 kB 22.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 146.7/146.7 kB 16.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 80.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.12.0 requires fsspec==2025.12.0, but you have fsspec 2025.3.0 which is incompatible.


In [ ]:
import json
import re
import gc
import copy
import random
from collections import Counter

import torch
import numpy as np
import pandas as pd
from datasets import load_dataset, Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader

import optipfair as opf

print(torch.cuda.get_device_name(0))

NVIDIA A100-SXM4-40GB


In [ ]:
import os
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

def set_seed(seed=42):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(True, warn_only=True)

set_seed(42)

## 2. Configuration

`RANDOM_STATE` and `TEST_SIZE` are set to match the M notebook. When
`model_m_handoff.json` is present, the next code cell verifies both values
before the dataset is rebuilt; the handoff stores the split parameters, not the
examples themselves.

This run removes three layers from M and asks OptiPFair to reduce the MLP width
using `EXPANSION_RATE=175` and divisor 128. The actual intermediate size is
reported after pruning because the library resolves the rate against the model
being pruned.

For KD, the code uses `T` as the teacher with `alpha=0.3`, `beta=0.6`,
`gamma=0.1`, `delta=0.0`, and temperature 1.25. The notebook is prepared to
switch the teacher by changing only `KD_TEACHER` to `"M"` or `"T"`; the current
configuration uses `"T"`. Three seeds are evaluated, and seed 42 is fixed in
advance as the canonical S checkpoint.

In [ ]:
DATASET_NAME = "Salesforce/xlam-function-calling-60k"
TEACHER_REPO_ID = "oopere/qwen3-0.6b-weather-geo-specialist"   # T, for reference only
MODEL_M_REPO_ID = "oopere/qwen3-0.6b-weather-geo-M"            # teacher for this step
MODEL_S_REPO_ID = "oopere/qwen3-0.6b-weather-geo-S"

# --- Data (must match the M notebook) ---
TEST_SIZE = 0.30
RANDOM_STATE = 42
MIN_EXAMPLES_PER_FUNCTION = 3
ALLOWED_FUNCTIONS = {"get_ip_zipcode", "get_city_from_zipcode", "local_weather_api"}

# --- Calibration ---
CALIBRATION_BATCH_SIZE = 8
MAX_CALIBRATION_LENGTH = 1152

# --- Depth pruning ---
NUM_LAYERS_TO_REMOVE = 3          # 25 -> 22

# --- Width pruning ---
# OptiPFair reads expansion_rate against the model being pruned, so this is not
# an absolute target: 175 here means 175/225 of M's 2304, not 175% of hidden.
# The cell below prints what actually landed -- check it and adjust.
EXPANSION_RATE = 175
EXPANSION_DIVISOR = 128

# --- Knowledge distillation ---
ALPHA = 0.3
BETA = 0.6
GAMMA = 0.1
DELTA = 0.0
TEMPERATURE = 1.25
SKEW_ALPHA = 0.4
EPOCHS = 16
LEARNING_RATE = 3e-5
DISTILL_BATCH_SIZE = 4
ACCUMULATION_STEPS = 4            # effective batch = 16
MAX_TRAIN_LENGTH = 1152
KD_TEACHER = "T"

# --- Seeds ---
SEEDS = [42, 43, 44]
CANONICAL_SEED = SEEDS[0]

PUSH_TO_HUB = False

In [ ]:
# Handoff from the M notebook, if present. It carries the split parameters and
# M's architecture so this notebook can check it is building on what it thinks.
try:
    with open("model_m_handoff.json") as f:
        handoff = json.load(f)
    print(json.dumps(handoff, indent=2))
    assert handoff["random_state"] == RANDOM_STATE, "split does not match the M notebook"
    assert handoff["test_size"] == TEST_SIZE, "split does not match the M notebook"
except FileNotFoundError:
    handoff = None
    print("No handoff file found -- make sure RANDOM_STATE and TEST_SIZE match "
          "the M notebook by hand.")

No handoff file found -- make sure RANDOM_STATE and TEST_SIZE match the M notebook by hand.


## 3. Dataset

In [ ]:
def curate_domain_dataset(dataset, allowed_functions=ALLOWED_FUNCTIONS,
                          min_examples=MIN_EXAMPLES_PER_FUNCTION):
    matched = [
        ex for ex in dataset
        if len(json.loads(ex["answers"])) == 1
        and json.loads(ex["answers"])[0]["name"] in allowed_functions
    ]
    function_counts = Counter(json.loads(ex["answers"])[0]["name"] for ex in matched)
    main_functions = {name for name, count in function_counts.items() if count >= min_examples}
    return [ex for ex in matched if json.loads(ex["answers"])[0]["name"] in main_functions]

raw_dataset = load_dataset(DATASET_NAME, split="train")
domain_examples = curate_domain_dataset(raw_dataset)
print(f"Curated domain examples: {len(domain_examples)}")

README.md:   0%|          | 0.00/10.7k [00:00<?, ?B/s]

xlam_function_calling_60k.json: reconstructing file:   0%|          |  0.00B / 96.1MB            

xlam_function_calling_60k.json: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/60000 [00:00<?, ? examples/s]

Curated domain examples: 233


In [ ]:
def normalize_query(text):
    return re.sub(r"[^\w\s]", "", text.lower()).strip()

def deduplicate_by_query(examples):
    seen = set()
    unique = []
    for ex in examples:
        key = normalize_query(ex["query"])
        if key not in seen:
            seen.add(key)
            unique.append(ex)
    return unique

domain_examples = deduplicate_by_query(domain_examples)
print(f"After deduplication: {len(domain_examples)}")

After deduplication: 177


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_M_REPO_ID)
tokenizer.padding_side = "right"
print("pad_token:", repr(tokenizer.pad_token), tokenizer.pad_token_id)
print("eos_token:", repr(tokenizer.eos_token), tokenizer.eos_token_id)

config.json:   0%|          | 0.00/1.35k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

chat_template.jinja:   0%|          | 0.00/4.17k [00:00<?, ?B/s]

pad_token: '<|endoftext|>' 151643
eos_token: '<|im_end|>' 151645


In [ ]:
def build_prompt_completion(example, tokenizer):
    tools = json.loads(example["tools"])
    answers = json.loads(example["answers"])
    user_message = {"role": "user", "content": example["query"]}
    assistant_message = {
        "role": "assistant",
        "content": None,
        "tool_calls": [{
            "type": "function",
            "function": {"name": answers[0]["name"], "arguments": answers[0]["arguments"]},
        }],
    }
    full_text = tokenizer.apply_chat_template(
        [user_message, assistant_message], tools=tools, tokenize=False, enable_thinking=False,
    )
    prompt_text = tokenizer.apply_chat_template(
        [user_message], tools=tools, tokenize=False, add_generation_prompt=True,
        enable_thinking=False,
    )
    assert full_text.startswith(prompt_text), "prompt is not a prefix of the full rendered text"
    return {"prompt": prompt_text, "completion": full_text[len(prompt_text):]}

function_labels = [json.loads(ex["answers"])[0]["name"] for ex in domain_examples]
train_examples, test_examples = train_test_split(
    domain_examples, test_size=TEST_SIZE, stratify=function_labels,
    random_state=RANDOM_STATE,
)
print(f"Train examples: {len(train_examples)}")
print(f"Test examples:  {len(test_examples)}")

train_dataset = Dataset.from_list([build_prompt_completion(ex, tokenizer) for ex in train_examples])
test_dataset = Dataset.from_list([build_prompt_completion(ex, tokenizer) for ex in test_examples])

Train examples: 123
Test examples:  54


In [ ]:
lengths = [
    len(tokenizer(ex["prompt"] + ex["completion"])["input_ids"])
    for ex in train_dataset
]
print(f"train prompt+completion -- min {min(lengths)}, mean {sum(lengths)//len(lengths)}, max {max(lengths)}")
over = sum(1 for l in lengths if l > MAX_TRAIN_LENGTH)
print(f"over MAX_TRAIN_LENGTH ({MAX_TRAIN_LENGTH}): {over} ({over/len(lengths)*100:.1f}%)")

train prompt+completion -- min 193, mean 395, max 1110
over MAX_TRAIN_LENGTH (1152): 0 (0.0%)


## 4. Models used in this step

M is loaded as the pruning source: its 25-layer topology and 2304-wide MLPs
are the starting point for S. T is loaded separately as the KD teacher because
`KD_TEACHER = "T"`; it is also retained as the original specialist reference.
The local `count_parameters` helper counts all parameters directly, so M does
not need to be frozen for the parameter report.

In [ ]:
gc.collect()
torch.cuda.empty_cache()

# M: pruning source. Its topology is what S inherits.
model_m = AutoModelForCausalLM.from_pretrained(
    MODEL_M_REPO_ID, dtype=torch.bfloat16,
).to("cuda")
model_m.generation_config.pad_token_id = tokenizer.pad_token_id
model_m.eval()

inter_m = Counter(l.mlp.gate_proj.out_features for l in model_m.model.layers)
assert len(inter_m) == 1, f"M has heterogeneous widths and will not serve: {dict(inter_m)}"
print(f"M loaded -- {model_m.config.num_hidden_layers} layers, "
      f"intermediate {max(inter_m)}")

# T: distillation teacher. Never pruned, never degraded.
teacher_model = AutoModelForCausalLM.from_pretrained(
    TEACHER_REPO_ID, dtype=torch.bfloat16,
).to("cuda")
teacher_model.generation_config.pad_token_id = tokenizer.pad_token_id
teacher_model.eval()
print(f"T loaded -- {teacher_model.config.num_hidden_layers} layers, "
      f"intermediate {teacher_model.config.intermediate_size}")

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.29GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/278 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/187 [00:00<?, ?B/s]

...

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/187 [00:00<?, ?B/s]

T loaded -- 28 layers, intermediate 3072


## 5. Evaluation harness

In [ ]:
def generate_tool_call(model, tokenizer, example, max_new_tokens=200):
    tools = json.loads(example["tools"])
    messages = [{"role": "user", "content": example["query"]}]
    prompt = tokenizer.apply_chat_template(
        messages, tools=tools, tokenize=False, add_generation_prompt=True,
        enable_thinking=False,
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=None, top_p=None, top_k=None,
        )
    return tokenizer.decode(output_ids[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

def parse_tool_call(generated_text):
    match = re.search(r"<tool_call>\s*(\{.*?\})\s*</tool_call>", generated_text, re.DOTALL)
    if not match:
        return None
    try:
        return json.loads(match.group(1))
    except json.JSONDecodeError:
        return None

In [ ]:
def is_substring_equivalent(a, b):
    a_clean = re.sub(r"[^\w\s]", "", str(a).lower()).strip()
    b_clean = re.sub(r"[^\w\s]", "", str(b).lower()).strip()
    if not a_clean or not b_clean:
        return False
    if not re.search(r"[a-z]", a_clean) or not re.search(r"[a-z]", b_clean):
        return False
    if len(a_clean) < 3 or len(b_clean) < 3:
        return a_clean == b_clean
    return a_clean in b_clean or b_clean in a_clean

def is_close_coordinate(key, predicted, expected, tolerance_degrees=0.05):
    if not any(k in key.lower() for k in ["lat", "lon", "lng"]):
        return False
    try:
        return abs(float(predicted) - float(expected)) <= tolerance_degrees
    except (TypeError, ValueError):
        return False

def value_matches(key, predicted, expected, tolerant):
    if tolerant:
        try:
            if float(predicted) == float(expected):
                return True
        except (TypeError, ValueError):
            pass
        if isinstance(predicted, str) and isinstance(expected, str):
            if predicted.strip().lower() == expected.strip().lower():
                return True
            if not re.search(r"[a-z]", (predicted + expected).lower()):
                if re.sub(r"\s+", "", predicted) == re.sub(r"\s+", "", expected):
                    return True
            if is_substring_equivalent(predicted, expected):
                return True
        if is_close_coordinate(key, predicted, expected):
            return True
    return predicted == expected

def call_matches(predicted, ground_truth, tolerant):
    if predicted is None or predicted.get("name") != ground_truth["name"]:
        return False
    predicted_args = predicted.get("arguments", {})
    expected_args = ground_truth["arguments"]
    if not tolerant and set(predicted_args.keys()) != set(expected_args.keys()):
        return False
    return all(
        key in predicted_args and value_matches(key, predicted_args[key], expected_value, tolerant)
        for key, expected_value in expected_args.items()
    )

In [ ]:
def evaluate_model(model, tokenizer, examples, verbose=True):
    results = []
    for example in examples:
        ground_truth = json.loads(example["answers"])[0]
        generated_text = generate_tool_call(model, tokenizer, example)
        predicted = parse_tool_call(generated_text)
        results.append({
            "generated_text": generated_text,
            "predicted": predicted,
            "ground_truth": ground_truth,
            "valid_json": predicted is not None,
            "exact_match": call_matches(predicted, ground_truth, tolerant=False),
            "tolerant_match": call_matches(predicted, ground_truth, tolerant=True),
        })
    n = len(results)
    if verbose:
        print(f"  Valid JSON:      {sum(r['valid_json'] for r in results) / n:.1%}")
        print(f"  Exact match:     {sum(r['exact_match'] for r in results) / n:.1%}")
        print(f"  Tolerant match:  {sum(r['tolerant_match'] for r in results) / n:.1%}")
    return results

def breakdown_mismatches(results, verbose=True):
    no_call = wrong_function = wrong_args = 0
    for result in results:
        if result["tolerant_match"]:
            continue
        predicted = result["predicted"]
        if predicted is None:
            no_call += 1
        elif predicted.get("name") != result["ground_truth"]["name"]:
            wrong_function += 1
        else:
            wrong_args += 1
    if verbose:
        print(f"  No tool_call emitted:        {no_call}")
        print(f"  Wrong function selected:     {wrong_function}")
        print(f"  Right function, wrong args:  {wrong_args}")
    return {"no_call": no_call, "wrong_function": wrong_function, "wrong_args": wrong_args}

def count_parameters(model):
    total = sum(p.numel() for p in model.parameters())
    emb = model.get_input_embeddings().weight.numel()
    head = 0 if model.config.tie_word_embeddings else model.get_output_embeddings().weight.numel()
    return {"total": total, "embeddings": emb + head, "non_embedding": total - emb - head}

In [ ]:
m_params = count_parameters(model_m)
print({k: f"{v/1e6:.1f}M" for k, v in m_params.items()})

print("\nM (teacher for this step) -- on test_examples:")
m_results = evaluate_model(model_m, tokenizer, test_examples)
m_breakdown = breakdown_mismatches(m_results)

{'total': '645.5M', 'embeddings': '155.6M', 'non_embedding': '489.9M'}

M (teacher for this step) -- on test_examples:
  Valid JSON:      100.0%
  Exact match:     96.3%
  Tolerant match:  96.3%
  No tool_call emitted:        0
  Wrong function selected:     0
  Right function, wrong args:  2


## 6. Calibration data

The calibration set contains prompt-only tool-calling contexts with dynamic
padding. Layer importance is measured on M, because M is the model whose depth
is being pruned. The same prompts and dataloader are then supplied to the width
pruning call, which operates on a copy of the depth-pruned M topology.

In [ ]:
calibration_texts = list(train_dataset["prompt"])

lens = [len(tokenizer(t)["input_ids"]) for t in calibration_texts]
print(f"calibration token lengths -- min {min(lens)}, mean {sum(lens)//len(lens)}, max {max(lens)}")
assert max(lens) <= MAX_CALIBRATION_LENGTH, (
    f"truncation would drop part of the prompt: raise MAX_CALIBRATION_LENGTH to >= {max(lens)}"
)

calibration_dataset = Dataset.from_dict({"text": calibration_texts})
calibration_dataset = calibration_dataset.map(
    lambda ex: tokenizer(
        ex["text"], truncation=True, max_length=MAX_CALIBRATION_LENGTH, return_tensors=None,
    ),
    batched=True,
)

def collate_calibration(batch):
    max_len = max(len(ex["input_ids"]) for ex in batch)
    pad_id = tokenizer.pad_token_id
    return {
        "input_ids": torch.tensor([
            ex["input_ids"] + [pad_id] * (max_len - len(ex["input_ids"])) for ex in batch
        ]),
        "attention_mask": torch.tensor([
            ex["attention_mask"] + [0] * (max_len - len(ex["attention_mask"])) for ex in batch
        ]),
    }

calibration_dataloader = DataLoader(
    calibration_dataset, batch_size=CALIBRATION_BATCH_SIZE, collate_fn=collate_calibration,
)

calibration token lengths -- min 165, mean 363, max 1074


Map:   0%|          | 0/123 [00:00<?, ? examples/s]

## 7. Depth pruning: M → 22 layers

Layer importance is scored on M using cosine distance between each block's input
and output. The selector scans the ascending ranking, protects layers 0-3 and
the final two layers, and avoids adjacent removals. Indices here are in M's
numbering (0-24).

After pruning, the notebook composes the surviving-layer map from S to M and,
when the M handoff is available, from S back to T.

In [ ]:
importance_scores = opf.analyze_layer_importance(
    model_m, calibration_dataloader, show_progress=True
)
sorted_layers = sorted(importance_scores.items(), key=lambda x: x[1])

print("Layers by importance (ascending -- most passive first):")
for layer_idx, score in sorted_layers:
    print(f"  Layer {layer_idx:2d}: {score:.6f}")

Processing batches: 100%|██████████| 16/16 [00:01<00:00,  8.28it/s]

Layers by importance (ascending -- most passive first):
  Layer 12: 0.039942
  Layer 13: 0.055461
  Layer 14: 0.058498
  Layer 21: 0.059133
  Layer 11: 0.067931
  Layer 22: 0.068203
  Layer 20: 0.073455
  Layer 17: 0.074485
  Layer 16: 0.077461
  Layer 10: 0.079689
  Layer 15: 0.084207
  Layer 19: 0.084592
  Layer 18: 0.085236
  Layer  5: 0.085456
  Layer  8: 0.086018
  Layer  9: 0.086620
  Layer  7: 0.088572
  Layer  6: 0.097538
  Layer 23: 0.100192
  Layer  4: 0.104269
  Layer  2: 0.108227
  Layer  3: 0.113059
  Layer 24: 0.115681
  Layer  1: 0.147197
  Layer  0: 0.938411


In [ ]:
def select_layers_to_prune(importance_scores, num_layers_to_remove,
                           heuristic_protection=True, adjacent_protection=True):
    num_layers = len(importance_scores)
    protected = {0, 1, 2, 3, num_layers - 2, num_layers - 1} if heuristic_protection else set()

    sorted_layers = sorted(importance_scores.items(), key=lambda x: x[1])
    selected = []
    for layer, score in sorted_layers:
        if heuristic_protection and layer in protected:
            continue
        if adjacent_protection and any(abs(layer - l) == 1 for l in selected):
            continue
        selected.append(layer)
        if len(selected) >= num_layers_to_remove:
            break

    return sorted(selected)

layers_to_remove = select_layers_to_prune(
    importance_scores, NUM_LAYERS_TO_REMOVE,
    heuristic_protection=True, adjacent_protection=True,
)
print(f"Layers selected for removal (M numbering): {layers_to_remove}")

if handoff is not None:
    m_map = {int(k): v for k, v in handoff["depth_index_map"].items()}
    print(f"Same layers in T numbering: {[m_map[i] for i in layers_to_remove]}")

Layers selected for removal (M numbering): [12, 14, 21]


In [ ]:
depth_pruned_model, depth_stats = opf.prune_model(
    model=copy.deepcopy(model_m),
    pruning_type="DEPTH",
    layer_indices=layers_to_remove,
    show_progress=True,
    return_stats=True,
)
depth_pruned_model.generation_config.pad_token_id = tokenizer.pad_token_id
depth_pruned_model.eval()

print({k: depth_stats[k] for k in [
    "original_layer_count", "final_layer_count", "layers_removed",
    "layer_reduction_percentage",
]})

Removing layers: 100%|██████████| 25/25 [00:00<00:00, 361577.93it/s]

{'original_layer_count': 25, 'final_layer_count': 22, 'layers_removed': 3, 'layer_reduction_percentage': 12.0}


In [ ]:
# Compose the index maps so S's layers can be traced all the way back to T.
surviving_in_m = [i for i in range(model_m.config.num_hidden_layers)
                  if i not in layers_to_remove]
s_to_m = {new: old for new, old in enumerate(surviving_in_m)}

if handoff is not None:
    m_to_t = {int(k): v for k, v in handoff["depth_index_map"].items()}
    s_to_t = {new: m_to_t[m_idx] for new, m_idx in s_to_m.items()}
    print("S index -> M index -> T index:")
    for new in sorted(s_to_m):
        print(f"  {new:2d} -> {s_to_m[new]:2d} -> {s_to_t[new]:2d}")
else:
    s_to_t = None
    print("S -> M index map:", s_to_m)

S -> M index map: {0: 0, 1: 1, 2: 2, 3: 3, 4: 4, 5: 5, 6: 6, 7: 7, 8: 8, 9: 9, 10: 10, 11: 11, 12: 13, 13: 15, 14: 16, 15: 17, 16: 18, 17: 19, 18: 20, 19: 22, 20: 23, 21: 24}


In [ ]:
print(f"After depth pruning ({depth_pruned_model.config.num_hidden_layers} layers) "
      "-- on test_examples:")
post_depth_results = evaluate_model(depth_pruned_model, tokenizer, test_examples)
post_depth_breakdown = breakdown_mismatches(post_depth_results)

After depth pruning (22 layers) -- on test_examples:
  Valid JSON:      94.4%
  Exact match:     7.4%
  Tolerant match:  13.0%
  No tool_call emitted:        3
  Wrong function selected:     32
  Right function, wrong args:  12


## 8. Width pruning

The width call uses PPM in hybrid mode through
`neuron_selection_method="MAW"`, with the calibration dataloader supplying
activation statistics. `EXPANSION_RATE=175` is resolved by OptiPFair against
the model being pruned, so it is not an absolute intermediate-size value or a
percentage of the hidden size.

The call prunes every MLP in a copy of the depth-pruned model to a uniform
width. The following cell reports the actual intermediate size, expansion over
the hidden state, divisor alignment, and parameter reduction before KD.

In [ ]:
num_layers_m = depth_pruned_model.config.num_hidden_layers
num_layers_m = depth_pruned_model.config.num_hidden_layers
print(f"Width pruning all {num_layers_m} layers to a uniform intermediate size")



Width pruning all 22 layers to a uniform intermediate size


In [ ]:
# deepcopy: prune_model mutates the model it receives, and the statistics
# helper compares against the object passed in.
model_s_pruned = opf.prune_model(
    model=copy.deepcopy(depth_pruned_model),
    pruning_type="MLP_GLU",
    neuron_selection_method="MAW",      # PPM; the only method supporting hybrid mode
    pruning_percentage=None,            # must be None when expansion_rate is set
    expansion_rate=EXPANSION_RATE,
    expansion_divisor=EXPANSION_DIVISOR,
    dataloader=calibration_dataloader,  # enables the data-driven hybrid ranking
    #layer_indices=width_layer_indices,
    show_progress=True,
    return_stats=False,
)
model_s_pruned.generation_config.pad_token_id = tokenizer.pad_token_id
model_s_pruned.eval()

Pruning all layers: 100%|██████████| 22/22 [00:00<00:00, 24.75it/s]


Qwen3ForCausalLM(
  (model): Qwen3Model(
    (embed_tokens): Embedding(151936, 1024, padding_idx=151643)
    (layers): ModuleList(
      (0-21): 22 x Qwen3DecoderLayer(
        (self_attn): Qwen3Attention(
          (q_proj): Linear(in_features=1024, out_features=2048, bias=False)
          (k_proj): Linear(in_features=1024, out_features=1024, bias=False)
          (v_proj): Linear(in_features=1024, out_features=1024, bias=False)
          (o_proj): Linear(in_features=2048, out_features=1024, bias=False)
          (q_norm): Qwen3RMSNorm((128,), eps=1e-06)
          (k_norm): Qwen3RMSNorm((128,), eps=1e-06)
        )
        (mlp): Qwen3MLP(
          (gate_proj): Linear(in_features=1024, out_features=1792, bias=False)
          (up_proj): Linear(in_features=1024, out_features=1792, bias=False)
          (down_proj): Linear(in_features=1792, out_features=1024, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen3RMSNorm((1024,), eps=1e-06)
        (

In [ ]:
# Check what landed before going any further.
inter_s = Counter(l.mlp.gate_proj.out_features for l in model_s_pruned.model.layers)
pruned_width = model_s_pruned.model.layers[0].mlp.gate_proj.out_features
hidden = model_s_pruned.config.hidden_size

print(f"layers: {model_s_pruned.config.num_hidden_layers}")
print(f"intermediate sizes: {dict(inter_s)}")
print(f"absolute expansion vs hidden ({hidden}): {pruned_width * 100 / hidden:.0f}%")
print(f"multiple of {EXPANSION_DIVISOR}: {pruned_width % EXPANSION_DIVISOR == 0}")

s_params = count_parameters(model_s_pruned)
print()
print({k: f"{v/1e6:.1f}M" for k, v in s_params.items()})
print(f"non-embedding vs M: {1 - s_params['non_embedding'] / m_params['non_embedding']:.1%} smaller")

layers: 22
intermediate sizes: {1792: 22}
absolute expansion vs hidden (1024): 175%
multiple of 128: True

{'total': '570.7M', 'embeddings': '155.6M', 'non_embedding': '415.2M'}
non-embedding vs M: 15.3% smaller


In [ ]:
print("After depth + width pruning, before KD -- on test_examples:")
pre_kd_results = evaluate_model(model_s_pruned, tokenizer, test_examples)
pre_kd_breakdown = breakdown_mismatches(pre_kd_results)

After depth + width pruning, before KD -- on test_examples:
  Valid JSON:      87.0%
  Exact match:     31.5%
  Tolerant match:  31.5%
  No tool_call emitted:        7
  Wrong function selected:     23
  Right function, wrong args:  7


## 9. Knowledge distillation: T → S

The student starts from the depth-and-width-pruned copy of M, but the current
configuration uses **T** as its teacher because `KD_TEACHER = "T"`. M remains
the structural source for S; it is also evaluated as the cascade reference.

Labels are completion-only masked, so the hard-label task loss supervises the
tool-call completion rather than the prompt. The run combines hard-label loss,
teacher-logit KL, and feature alignment with `alpha=0.3`, `beta=0.6`, and
`gamma=0.1`. Three seeds use the same configuration, with seed 42 fixed in
advance as the canonical checkpoint rather than selected from test scores.

In [ ]:
def build_distillation_example(prompt, completion, tokenizer, max_length=MAX_TRAIN_LENGTH):
    full_text = prompt + completion
    full_encoded = tokenizer(full_text, truncation=True, max_length=max_length)
    prompt_ids = tokenizer(prompt, truncation=True, max_length=max_length)["input_ids"]
    prompt_len = len(prompt_ids)

    labels = list(full_encoded["input_ids"])
    for i in range(min(prompt_len, len(labels))):
        labels[i] = -100

    return {
        "input_ids": full_encoded["input_ids"],
        "attention_mask": full_encoded["attention_mask"],
        "labels": labels,
    }

distillation_examples = [
    build_distillation_example(ex["prompt"], ex["completion"], tokenizer)
    for ex in train_dataset
]
distillation_dataset = Dataset.from_list(distillation_examples)

unsupervised = [
    i for i, ex in enumerate(distillation_examples)
    if all(l == -100 for l in ex["labels"])
]
assert not unsupervised, (
    f"examples with no supervised token: {unsupervised} -- raise MAX_TRAIN_LENGTH"
)
print(f"{len(distillation_dataset)} distillation examples, all with supervised targets.")

def collate_distillation(batch):
    max_len = max(len(ex["input_ids"]) for ex in batch)
    input_ids, attention_mask, labels = [], [], []
    for ex in batch:
        pad_len = max_len - len(ex["input_ids"])
        input_ids.append(ex["input_ids"] + [tokenizer.pad_token_id] * pad_len)
        attention_mask.append(ex["attention_mask"] + [0] * pad_len)
        labels.append(ex["labels"] + [-100] * pad_len)
    return {
        "input_ids": torch.tensor(input_ids),
        "attention_mask": torch.tensor(attention_mask),
        "labels": torch.tensor(labels),
    }

distillation_dataloader = DataLoader(
    distillation_dataset, batch_size=DISTILL_BATCH_SIZE, shuffle=True,
    collate_fn=collate_distillation,
)

123 distillation examples, all with supervised targets.


In [ ]:
seed_runs = {}

for seed in SEEDS:
    print(f"\n{'=' * 60}\nseed {seed}\n{'=' * 60}")
    set_seed(seed)

    distilled, stats = opf.distill_model(
        student_model=copy.deepcopy(model_s_pruned),
        teacher_model=teacher_model if KD_TEACHER == "T" else model_m,
        dataloader=distillation_dataloader,
        alpha=ALPHA,
        beta=BETA,
        gamma=GAMMA,
        delta=DELTA,
        temperature=TEMPERATURE,
        skew_alpha=SKEW_ALPHA,
        epochs=EPOCHS,
        learning_rate=LEARNING_RATE,
        accumulation_steps=ACCUMULATION_STEPS,
        show_progress=True,
        return_stats=True,
    )
    distilled.generation_config.pad_token_id = tokenizer.pad_token_id
    distilled.eval()

    results = evaluate_model(distilled, tokenizer, test_examples, verbose=False)
    exact = sum(r["exact_match"] for r in results) / len(results)
    print(f"  exact match: {exact:.1%}")

    seed_runs[seed] = {"results": results, "stats": stats, "exact_match": exact}

    # S is the canonical seed's model, fixed in advance -- not the best of the
    # three. Picking the best run after seeing the scores is selecting on the
    # test set.
    if seed == CANONICAL_SEED:
        model_s = distilled
        distill_stats = stats
    else:
        del distilled
        gc.collect()
        torch.cuda.empty_cache()


seed 42


Epoch 1/16:   0%|          | 0/31 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/torch/autograd/graph.py:869: UserWarning: Memory Efficient attention defaults to a non-deterministic algorithm. To explicitly enable determinism call torch.use_deterministic_algorithms(True, warn_only=False). (Triggered internally at /pytorch/aten/src/ATen/native/transformers/cuda/attention_backward.cu:900.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


Epoch 2/16:   0%|          | 0/31 [00:00<?, ?it/s]

Epoch 3/16:   0%|          | 0/31 [00:00<?, ?it/s]

...

Epoch 13/16:   0%|          | 0/31 [00:00<?, ?it/s]

Epoch 14/16:   0%|          | 0/31 [00:00<?, ?it/s]

Epoch 15/16:   0%|          | 0/31 [00:00<?, ?it/s]

Epoch 16/16:   0%|          | 0/31 [00:00<?, ?it/s]

  exact match: 92.6%


In [ ]:
scores = [seed_runs[s]["exact_match"] for s in SEEDS]
spread = max(scores) - min(scores)

variance_df = pd.DataFrame(
    [{"seed": str(s), "exact_match": f"{seed_runs[s]['exact_match']:.1%}"} for s in SEEDS]
    + [
        {"seed": "mean", "exact_match": f"{sum(scores)/len(scores):.1%}"},
        {"seed": "range", "exact_match": f"{spread*100:.1f} pp"},
    ]
)
print(f"Spread across seeds: {spread*len(test_examples):.0f} of {len(test_examples)} "
      "test examples")
variance_df

Spread across seeds: 2 of 54 test examples


,seed,exact_match
0,42,94.4%
1,43,90.7%
2,44,92.6%
3,mean,92.6%
4,range,3.7 pp


In [ ]:
loss_total = distill_stats["loss_history"]["total"]
loss_task = distill_stats["loss_history"]["task"]
loss_logits = distill_stats["loss_history"]["logits"]

print(f"seed {CANONICAL_SEED}")
print(f"{'epoch':>6}  {'total':>8}  {'task':>8}  {'logits':>8}")
for i, (t, ta, lo) in enumerate(zip(loss_total, loss_task, loss_logits), 1):
    print(f"{i:>6}  {t:>8.4f}  {ta:>8.4f}  {lo:>8.4f}")

# KL is non-negative by definition. A negative value means the student has
# converged close enough to the teacher that numerical error dominates the
# term -- beta is no longer contributing anything at that point.
if min(loss_logits) < 0:
    print("\nWARNING: logits loss went negative -- the KL term has collapsed into "
          "numerical noise. Consider raising temperature.")

best = min(range(len(loss_total)), key=lambda i: loss_total[i]) + 1
print(f"\nlowest total loss at epoch {best} of {len(loss_total)}")

seed 42
 epoch     total      task    logits
     1    0.0775    0.0969    0.0517
     2    0.0336    0.0321    0.0126
     3    0.0225    0.0090    0.0071
     4    0.0173    0.0030    0.0024
     5    0.0158    0.0012    0.0016
     6    0.0153    0.0005    0.0015
     7    0.0150    0.0003    0.0013
     8    0.0150    0.0003    0.0014
     9    0.0149    0.0003    0.0015
    10    0.0148    0.0002    0.0014
    11    0.0149    0.0002    0.0015
    12    0.0147    0.0002    0.0013
    13    0.0148    0.0002    0.0014
    14    0.0148    0.0002    0.0015
    15    0.0147    0.0002    0.0014
    16    0.0148    0.0002    0.0014

lowest total loss at epoch 12 of 16


## 10. Results

In [ ]:
post_kd_results = seed_runs[CANONICAL_SEED]["results"]
n = len(post_kd_results)
print(f"S ({model_s.config.num_hidden_layers} layers, seed {CANONICAL_SEED}) "
      "-- on test_examples:")
print(f"  Valid JSON:      {sum(r['valid_json'] for r in post_kd_results) / n:.1%}")
print(f"  Exact match:     {sum(r['exact_match'] for r in post_kd_results) / n:.1%}")
print(f"  Tolerant match:  {sum(r['tolerant_match'] for r in post_kd_results) / n:.1%}")
post_kd_breakdown = breakdown_mismatches(post_kd_results)

S (22 layers, seed 42) -- on test_examples:
  Valid JSON:      100.0%
  Exact match:     94.4%
  Tolerant match:  94.4%
  No tool_call emitted:        0
  Wrong function selected:     0
  Right function, wrong args:  3


In [ ]:
def summarize(results, breakdown, label):
    n = len(results)
    return {
        "model": label,
        "valid_json": sum(r["valid_json"] for r in results) / n,
        "exact_match": sum(r["exact_match"] for r in results) / n,
        "tolerant_match": sum(r["tolerant_match"] for r in results) / n,
        "no_call": breakdown["no_call"],
        "wrong_function": breakdown["wrong_function"],
        "wrong_args": breakdown["wrong_args"],
    }

comparison_df = pd.DataFrame([
    summarize(m_results, m_breakdown, f"M ({model_m.config.num_hidden_layers}L) -- teacher"),
    summarize(post_depth_results, post_depth_breakdown,
              f"depth only ({depth_pruned_model.config.num_hidden_layers}L)"),
    summarize(pre_kd_results, pre_kd_breakdown, "depth+width, pre-KD"),
    summarize(post_kd_results, post_kd_breakdown,
              f"S, post-KD ({model_s.config.num_hidden_layers}L)"),
])
comparison_df

,model,valid_json,exact_match,tolerant_match,no_call,wrong_function,wrong_args
0,M (25L) -- teacher,1.000000,0.962963,0.962963,0,0,2
1,depth only (22L),0.944444,0.074074,0.129630,3,32,12
2,"depth+width, pre-KD",0.870370,0.314815,0.314815,7,23,7
3,"S, post-KD (22L)",1.000000,0.944444,0.944444,0,0,3


In [ ]:
function_results = {}
for ex, result in zip(test_examples, post_kd_results):
    fn_name = json.loads(ex["answers"])[0]["name"]
    function_results.setdefault(fn_name, []).append(result["exact_match"])

for fn_name, matches in function_results.items():
    print(f"{fn_name}: {sum(matches)}/{len(matches)} exact match")

local_weather_api: 8/10 exact match
get_ip_zipcode: 25/26 exact match
get_city_from_zipcode: 18/18 exact match


In [ ]:
# Does S fail where M succeeded? In a cascade this is the question that
# matters: errors accumulate down the chain.
m_wrong = {i for i, r in enumerate(m_results) if not r["tolerant_match"]}
s_wrong = {i for i, r in enumerate(post_kd_results) if not r["tolerant_match"]}

print(f"M failures: {len(m_wrong)}   S failures: {len(s_wrong)}")
print(f"Extra failures vs M: {len(s_wrong - m_wrong)}")
for i in sorted(s_wrong - m_wrong):
    print("-" * 50)
    print("QUERY:       ", test_examples[i]["query"])
    print("GENERATED:   ", post_kd_results[i]["generated_text"])
    print("GROUND TRUTH:", post_kd_results[i]["ground_truth"])

M failures: 2   S failures: 3
Extra failures vs M: 1
--------------------------------------------------
QUERY:        What are the weather conditions and a 4-day forecast for Los Angeles, without air quality data, in Spanish?
GENERATED:    <tool_call>
{"name": "local_weather_api", "arguments": {"q": "Los Angeles", "aqi": "no", "lang": "fr", "num_of_days": 4}}
</tool_call>
GROUND TRUTH: {'name': 'local_weather_api', 'arguments': {'q': 'Los Angeles', 'num_of_days': 4, 'aqi': 'no', 'lang': 'es'}}


In [ ]:
def bootstrap_ci(flags, n_boot=10000, alpha=0.05, seed=42):
    rng = np.random.default_rng(seed)
    flags = np.asarray(flags, dtype=float)
    means = flags[rng.integers(0, len(flags), size=(n_boot, len(flags)))].mean(axis=1)
    return flags.mean(), np.quantile(means, alpha / 2), np.quantile(means, 1 - alpha / 2)

for label, results in [
    ("M      ", m_results),
    ("pre-KD ", pre_kd_results),
    ("S      ", post_kd_results),
]:
    mean, lo, hi = bootstrap_ci([r["exact_match"] for r in results])
    print(f"{label} exact match: {mean:.1%}  95% CI [{lo:.1%}, {hi:.1%}]")

print("\nThese intervals are wide because the test set is small. Differences of "
      "two or three examples are not resolvable here, and no configuration was "
      "chosen by comparing single runs against this set.")

M       exact match: 96.3%  95% CI [90.7%, 100.0%]
pre-KD  exact match: 31.5%  95% CI [18.5%, 44.4%]
S       exact match: 94.4%  95% CI [87.0%, 100.0%]

These intervals are wide because the test set is small. Differences of two or three examples are not resolvable here, and no configuration was chosen by comparing single runs against this set.


In [ ]:
family_df = pd.DataFrame([
    {
        "model": "M",
        "layers": model_m.config.num_hidden_layers,
        "intermediate": max(inter_m),
        "expansion_vs_hidden": f"{max(inter_m) * 100 // model_m.config.hidden_size}%",
        "non_emb_M": round(m_params["non_embedding"] / 1e6, 1),
        "total_M": round(m_params["total"] / 1e6, 1),
        "emb_pct": f"{m_params['embeddings'] / m_params['total']:.0%}",
        "kd_teacher": "T",
    },
    {
        "model": "S",
        "layers": model_s.config.num_hidden_layers,
        "intermediate": max(inter_s),
        "expansion_vs_hidden": f"{max(inter_s) * 100 // model_s.config.hidden_size}%",
        "non_emb_M": round(s_params["non_embedding"] / 1e6, 1),
        "total_M": round(s_params["total"] / 1e6, 1),
        "emb_pct": f"{s_params['embeddings'] / s_params['total']:.0%}",
        "kd_teacher": "M",
    },
])
family_df

,model,layers,intermediate,expansion_vs_hidden,non_emb_M,total_M,emb_pct,kd_teacher
0,M,25,2304,225%,489.9,645.5,24%,T
1,S,22,1792,175%,415.2,570.7,27%,M


The `emb_pct` column shows how the unchanged tied embedding table occupies an
ever larger share of each smaller model. The architecture changes remove
non-embedding parameters through depth and MLP pruning, so the reported
non-embedding counts are the clearest comparison of removed model capacity.

## 11. Save model S

In [ ]:
OUTPUT_DIR = "./model_s"
model_s.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"Saved to {OUTPUT_DIR}")

if PUSH_TO_HUB:
    from huggingface_hub import login
    login()
    model_s.push_to_hub(MODEL_S_REPO_ID, private=True)
    tokenizer.push_to_hub(MODEL_S_REPO_ID, private=True)
    print(f"Pushed to https://huggingface.co/{MODEL_S_REPO_ID}")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved to ./model_s


In [ ]:
handoff_s = {
    "model_s_repo": MODEL_S_REPO_ID,
    "layers": model_s.config.num_hidden_layers,
    "intermediate_size": max(inter_s),
    "expansion_rate_used": EXPANSION_RATE,
    "layers_removed_from_M": layers_to_remove,
    "s_to_m_index_map": s_to_m,
    "s_to_t_index_map": s_to_t,
    "random_state": RANDOM_STATE,
    "test_size": TEST_SIZE,
    "seed_scores": {str(s): seed_runs[s]["exact_match"] for s in SEEDS},
    "mean_exact_match": sum(scores) / len(scores),
}
with open("model_s_handoff.json", "w") as f:
    json.dump(handoff_s, f, indent=2)
print(json.dumps(handoff_s, indent=2))

{
  "model_s_repo": "oopere/qwen3-0.6b-weather-geo-S",
  "layers": 22,
  "intermediate_size": 1792,
  "expansion_rate_used": 175,
  "layers_removed_from_M": [
    12,
    14,
    21
  ],
  "s_to_m_index_map": {
    "0": 0,
    "1": 1,
    "2": 2,
    "3": 3,
    "4": 4,
    "5": 5,
    "6": 6,
    "7": 7,
    "8": 8,
    "9": 9,
    "10": 10,
    "11": 11,
    "12": 13,
    "13": 15,
    "14": 16,
    "15": 17,
    "16": 18,
    "17": 19,
    "18": 20,
    "19": 22,
    "20": 23,
    "21": 24
  },
  "s_to_t_index_map": null,
  "random_state": 42,
  "test_size": 0.3,
  "seed_scores": {
    "42": 0.9444444444444444,
    "43": 0.9074074074074074,
    "44": 0.9259259259259259
  },
  "mean_exact_match": 0.9259259259259259
}
